# Phase 6A D2 Refined Probe - Unit-of-Observation Re-examination

The first-pass D2 probe (notebook 05) tripped its escalation threshold (7.9% of rows in fold-spanning identical-profile clusters) using a coarse match key (School, Position, role, rounded Height/Weight). That key conflates genuine duplicate records with distinct same-program athletes of similar build.

This read-only notebook tightens the match key in tiers — from the original loose key down to a full-precision signature on all nine measurements plus missingness pattern — to measure how fast the "duplicate" count collapses and isolate near-certain duplicate records. It informs the dormant grouped-CV question for the Phase 6A acceptance record; it changes no protocol and starts no Phase 7.


## Boundaries

- Read-only refinement of the Phase 6A **D2** unit-of-observation probe. **No model, no features, no CV run, no protocol change.**
- No Phase 7, no HPO, no submissions, no model-family comparison.
- Untouched: `logs/experiment_log.csv` (candidate row only), `data/input/`, `notebooks/_official/`, `references/`, `outputs/submissions/`, `.vscode/settings.json`.
- `School` is used for duplicate detection only; it is never a model feature.
- Pre-write guards refuse to overwrite existing artifacts; the main log is asserted unchanged after writing.
- Frozen folds are read only to label which CV fold each row sits in; they are never recomputed or modified.


In [1]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import platform
import subprocess
import sys

import numpy as np
import pandas as pd


# ----------------------------------------------------------------------------
# Configuration and paths (read-only diagnostic: no model, no features)
# ----------------------------------------------------------------------------
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "input" / "train.csv").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root containing data/input/train.csv")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
EXPERIMENT_ID = "phase6a_d2_refined_probe_v1"
ID_COL = "Id"
TARGET_COL = "Drafted"
N_SPLITS = 5
ESCALATION_FRAC = 0.02
MEASUREMENTS = ["Height", "Weight", "Age", "Sprint_40yd", "Vertical_Jump",
                "Bench_Press_Reps", "Broad_Jump", "Agility_3cone", "Shuttle"]

TRAIN_PATH = PROJECT_ROOT / "data" / "input" / "train.csv"
FROZEN_FOLDS_PATH = PROJECT_ROOT / "outputs" / "folds" / "phase6_rf_sanity_baseline_v1_fold_assignments.csv"
MAIN_LOG_PATH = PROJECT_ROOT / "logs" / "experiment_log.csv"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "06_phase6a_d2_refined_probe.ipynb"

OUT_VAL = PROJECT_ROOT / "outputs" / "validation"
OUT_REP = PROJECT_ROOT / "outputs" / "reports"
TIER_SUMMARY_PATH = OUT_VAL / "phase6a_d2_refined_tier_summary.csv"
DUP_CANDIDATES_PATH = OUT_VAL / "phase6a_d2_refined_duplicate_candidates.csv"
REPORT_PATH = OUT_REP / "phase6a_d2_refined_probe_report.md"
LOG_CANDIDATE_PATH = OUT_REP / "phase6a_d2_refined_experiment_log_candidate.csv"
EXPECTED_ARTIFACTS = [TIER_SUMMARY_PATH, DUP_CANDIDATES_PATH, REPORT_PATH, LOG_CANDIDATE_PATH]

existing = [str(p.relative_to(PROJECT_ROOT)) for p in EXPECTED_ARTIFACTS if p.exists()]
if existing:
    raise FileExistsError("Refusing to overwrite existing refined-D2 artifacts: " + "; ".join(existing))

for required in [TRAIN_PATH, FROZEN_FOLDS_PATH, MAIN_LOG_PATH]:
    if not required.exists():
        raise FileNotFoundError(f"Missing required input: {required}")

train = pd.read_csv(TRAIN_PATH)
frozen_folds = pd.read_csv(FROZEN_FOLDS_PATH)
main_log_before = MAIN_LOG_PATH.read_text(encoding="utf-8")

if list(frozen_folds.columns) != [ID_COL, "fold"] or len(frozen_folds) != len(train):
    raise ValueError("Frozen folds integrity check failed")
if not frozen_folds[ID_COL].reset_index(drop=True).equals(train[ID_COL].reset_index(drop=True)):
    raise ValueError("Frozen folds Id order mismatch")
fold_of = frozen_folds.set_index(ID_COL)["fold"]
n_train = len(train)
print(f"Train rows: {n_train}; measurement coverage (non-null):")
print(train[MEASUREMENTS].notna().sum().to_string())


# ----------------------------------------------------------------------------
# Tiered identical-profile analysis (progressively stricter match keys)
# ----------------------------------------------------------------------------
train = train.copy()
train["Height_r1"] = train["Height"].round(1)
train["Weight_r0"] = train["Weight"].round(0)


def analyze(key_cols, label, description, require_nonnull=None):
    work = train
    if require_nonnull:
        mask = np.ones(len(train), dtype=bool)
        for c in require_nonnull:
            mask &= train[c].notna().to_numpy()
        work = train[mask]
    clusters = []
    for _, idx in work.groupby(key_cols, dropna=False).groups.items():
        sub = work.loc[idx]
        if len(sub) >= 2:
            folds_spanned = int(fold_of.loc[sub[ID_COL].to_numpy()].nunique())
            n_years = int(sub["Year"].nunique())
            clusters.append((sub, folds_spanned, n_years))
    n_rows = int(sum(len(s) for s, _, _ in clusters))
    fold_span_rows = int(sum(len(s) for s, fs, _ in clusters if fs >= 2))
    cross_year_rows = int(sum(len(s) for s, _, ny in clusters if ny >= 2))
    summary = {
        "tier": label,
        "description": description,
        "match_key": " + ".join(key_cols),
        "eligible_rows": int(len(work)),
        "n_clusters": len(clusters),
        "n_rows_in_clusters": n_rows,
        "pct_train": round(n_rows / n_train, 4),
        "fold_spanning_rows": fold_span_rows,
        "pct_fold_spanning": round(fold_span_rows / n_train, 4),
        "same_year_rows": n_rows - cross_year_rows,
        "cross_year_rows": cross_year_rows,
    }
    return summary, clusters


tier_specs = [
    (["School", "Position", "Player_Type", "Position_Type", "Height_r1", "Weight_r0"], "T1_loose",
     "Original coarse heuristic (Height~0.1, Weight~1)", None),
    (["School", "Position", "Height", "Weight"], "T2_exact_build",
     "Exact full-precision Height + Weight", None),
    (["School", "Position", "Height", "Weight", "Age"], "T3_build_age",
     "T2 + exact Age (Age present only)", ["Age"]),
    (["School", "Position", *MEASUREMENTS], "T4_full_signature",
     "School + Position + all 9 measurements + identical missingness pattern", None),
    (["Position", *MEASUREMENTS], "T4b_full_sig_no_school",
     "Full measurement signature ignoring School (catches transfers/typos)", None),
]

tier_rows = []
clusters_by_tier = {}
for key_cols, label, desc, req in tier_specs:
    summary, clusters = analyze(key_cols, label, desc, req)
    tier_rows.append(summary)
    clusters_by_tier[label] = clusters
    print(f"{label:24s} clusters={summary['n_clusters']:3d} rows={summary['n_rows_in_clusters']:4d} "
          f"({summary['pct_train']:.2%})  fold_spanning={summary['fold_spanning_rows']:4d} "
          f"({summary['pct_fold_spanning']:.2%})  same_year={summary['same_year_rows']} cross_year={summary['cross_year_rows']}")

tier_df = pd.DataFrame(tier_rows)


# ----------------------------------------------------------------------------
# Strict duplicate candidates = T4 (full measurement signature)
# ----------------------------------------------------------------------------
strict_clusters = clusters_by_tier["T4_full_signature"]
dup_rows = []
cluster_id = 0
for sub, folds_spanned, n_years in strict_clusters:
    cluster_id += 1
    n_nonnull_meas = int(sub[MEASUREMENTS].notna().all(axis=0).sum())  # measurements present for the whole cluster
    for _, r in sub.sort_values("Year").iterrows():
        dup_rows.append({
            "cluster_id": cluster_id,
            "cluster_size": int(len(sub)),
            "n_folds_spanned": folds_spanned,
            "n_distinct_years": n_years,
            "shared_nonnull_measurements": n_nonnull_meas,
            "same_year": bool(n_years == 1),
            ID_COL: int(r[ID_COL]),
            "fold": int(fold_of.loc[r[ID_COL]]),
            "Year": int(r["Year"]),
            "School": r["School"],
            "Position": r["Position"],
            "Height": float(r["Height"]),
            "Weight": float(r["Weight"]),
            "Age": (float(r["Age"]) if pd.notna(r["Age"]) else np.nan),
            "Drafted": int(r[TARGET_COL]),
        })
dup_candidates = pd.DataFrame(dup_rows)

t1 = tier_df.loc[tier_df["tier"] == "T1_loose"].iloc[0]
t4 = tier_df.loc[tier_df["tier"] == "T4_full_signature"].iloc[0]
strict_fold_span_rows = int(t4["fold_spanning_rows"])
strict_pct_fold_span = strict_fold_span_rows / n_train
refined_escalation = strict_pct_fold_span > ESCALATION_FRAC
if len(dup_candidates):
    strict_same_year_rows = int((dup_candidates["same_year"]).sum())
    strict_cross_year_rows = int((~dup_candidates["same_year"]).sum())
else:
    strict_same_year_rows = strict_cross_year_rows = 0

print(f"\nCollapse T1 -> T4 fold-spanning rows: {int(t1['fold_spanning_rows'])} -> {strict_fold_span_rows} "
      f"({t1['pct_fold_spanning']:.2%} -> {strict_pct_fold_span:.2%})")
print(f"Strict (T4) duplicates: same_year_rows={strict_same_year_rows} cross_year_rows={strict_cross_year_rows}")
print(f"Refined D2 escalation (strict fold-spanning > {ESCALATION_FRAC:.0%}): {refined_escalation}")


# ----------------------------------------------------------------------------
# Report + candidate log + write
# ----------------------------------------------------------------------------
def get_git_status() -> str:
    try:
        commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip()
        porcelain = subprocess.check_output(["git", "status", "--short"], cwd=PROJECT_ROOT, text=True).strip()
        return f"{commit} ({'dirty' if porcelain else 'clean'})"
    except Exception as exc:  # noqa: BLE001
        return f"git_status_unavailable: {exc}"


def markdown_table(df: pd.DataFrame, digits: int = 4) -> str:
    if df.empty:
        return "_No rows._"
    cols = [str(c) for c in df.columns]

    def fmt(v):
        if pd.isna(v):
            return ""
        if isinstance(v, (float, np.floating)):
            return f"{float(v):.{digits}f}"
        return str(v).replace("\n", " ").replace("|", "\\|")

    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(fmt(row[c]) for c in df.columns) + " |")
    return "\n".join(lines)


examples = dup_candidates.sort_values(["cluster_size", "cluster_id"], ascending=[False, True]).head(24) if len(dup_candidates) else dup_candidates

verdict = (
    "REFINED ESCALATION STANDS: a material share of rows are strict full-signature duplicates spanning CV folds; "
    "grouped CV is a live protocol question for human ruling."
    if refined_escalation else
    "REFINED ESCALATION CLEARED: under strict full-measurement-signature matching, fold-spanning duplicate rows fall "
    f"below the {ESCALATION_FRAC:.0%} threshold. The original T1 escalation was driven by coarse matching of distinct "
    "same-program/position athletes with similar builds, not genuine duplicate records. Recommendation: keep the frozen "
    "StratifiedKFold folds; record unit-of-observation as 'no material same-athlete duplication confirmed'."
)

report = f"""# Phase 6A D2 Refined Probe - Unit-of-Observation Re-examination

## Purpose

The first-pass D2 probe (notebook 05) flagged 7.9% of rows as fold-spanning identical-profile clusters and tripped the
escalation threshold. That heuristic matched only on (School, Position, role, coarse Height/Weight), which conflates
genuine duplicate records with distinct same-program athletes of similar build. This refined, read-only probe tightens
the match key step by step to separate the two. No model, no features, no protocol change.

## Environment

| Item | Value |
|---|---|
| Git status | {get_git_status()} |
| Python | {sys.version.split()[0]} |
| numpy | {np.__version__} |
| pandas | {pd.__version__} |
| Platform | {platform.platform()} |

## Tiered identical-profile analysis

Each tier requires rows to match on a progressively stricter key. `fold_spanning_rows` = rows in clusters whose members
fall in >= 2 CV folds (the only configuration that actually leaks identity across folds).

{markdown_table(tier_df)}

Collapse of fold-spanning rows from the loose to the strict key: **{int(t1['fold_spanning_rows'])} ({t1['pct_fold_spanning']:.2%}) -> {strict_fold_span_rows} ({strict_pct_fold_span:.2%})**.

## Strict duplicates (T4 full measurement signature)

A T4 cluster shares School + Position + identical values across all of {", ".join(MEASUREMENTS)} (including identical
missingness pattern). Two distinct athletes matching the entire measured vector is statistically implausible, so T4
clusters are near-certain duplicate records.

| Metric | Value |
|---|---|
| T4 clusters | {int(t4['n_clusters'])} |
| T4 rows | {int(t4['n_rows_in_clusters'])} ({t4['pct_train']:.2%} of train) |
| T4 fold-spanning rows | {strict_fold_span_rows} ({strict_pct_fold_span:.2%} of train) |
| T4 same-year rows | {strict_same_year_rows} |
| T4 cross-year rows | {strict_cross_year_rows} |
| Escalation threshold | > {ESCALATION_FRAC:.0%} of train |
| **Refined D2 escalation** | **{refined_escalation}** |

### Strict duplicate examples

{markdown_table(examples)}

## Verdict

{verdict}

Effect on prior results: none. Any residual identity leak is common-mode across all V0-V7/D1 variants on the shared
frozen folds and cancels in the paired deltas, so the gap decomposition and the D1 noise floor are unaffected. The
unit-of-observation question bears only on the absolute anchor level and Phase 7 generalization estimates.

## Next gate

Do not start Phase 7 until the Phase 6A acceptance record is signed. This refined probe is read-only evidence for that
acceptance decision; it changes no protocol on its own.
"""

candidate_log = pd.DataFrame([{
    "experiment_id": EXPERIMENT_ID,
    "date": datetime.now().strftime("%Y-%m-%d"),
    "phase": "Phase 6A - D2 refined probe",
    "notebook_or_script": str(NOTEBOOK_PATH.relative_to(PROJECT_ROOT)).replace("\\", "/"),
    "git_commit_or_status": get_git_status(),
    "data_version": "official_data_input_csvs",
    "fold_strategy": "loaded_frozen_phase6_folds (read-only; no CV run)",
    "random_seed": "n/a (no model)",
    "feature_block": "phase6a_d2_refined_duplicate_probe",
    "preprocessing_summary": "read-only tiered identical-profile match; no training",
    "model_family": "none",
    "model_params_summary": "n/a",
    "hpo_status": "not_run",
    "cv_auc_mean": np.nan,
    "cv_auc_std": np.nan,
    "fold_scores": "n/a",
    "slice_metrics_available": False,
    "leakage_checks_passed": "read_only_probe; main_log_unchanged; no_submission; School_diagnostic_only",
    "submission_created": False,
    "submission_path": None,
    "public_lb_score_if_submitted": None,
    "notes": (
        f"T1 fold-span {int(t1['fold_spanning_rows'])} ({t1['pct_fold_spanning']:.2%}) -> "
        f"T4 fold-span {strict_fold_span_rows} ({strict_pct_fold_span:.2%}); refined_escalation={refined_escalation}; "
        f"T4 same_year={strict_same_year_rows} cross_year={strict_cross_year_rows}"
    ),
    "decision": "ready_for_phase6a_human_review",
}])

existing_now = [str(p.relative_to(PROJECT_ROOT)) for p in EXPECTED_ARTIFACTS if p.exists()]
if existing_now:
    raise FileExistsError("Refusing to overwrite existing refined-D2 artifacts before write: " + "; ".join(existing_now))
for directory in [OUT_VAL, OUT_REP]:
    directory.mkdir(parents=True, exist_ok=True)

tier_df.to_csv(TIER_SUMMARY_PATH, index=False)
dup_candidates.to_csv(DUP_CANDIDATES_PATH, index=False)
REPORT_PATH.write_text(report, encoding="utf-8")
candidate_log.to_csv(LOG_CANDIDATE_PATH, index=False)

main_log_after = MAIN_LOG_PATH.read_text(encoding="utf-8")
if main_log_before != main_log_after:
    raise RuntimeError("logs/experiment_log.csv changed during refined D2 probe, which is forbidden")

print("\nArtifacts written:")
for p in EXPECTED_ARTIFACTS:
    print(" ", p.relative_to(PROJECT_ROOT), p.exists())
print("Main log unchanged:", main_log_before == main_log_after)
print("Refined D2 escalation:", refined_escalation)


Train rows: 2781; measurement coverage (non-null):
Height              2781
Weight              2781
Age                 2346
Sprint_40yd         2636
Vertical_Jump       2227
Bench_Press_Reps    2060
Broad_Jump          2200
Agility_3cone       1811
Shuttle             1869


T1_loose                 clusters=119 rows= 252 (9.06%)  fold_spanning= 220 (7.91%)  same_year=8 cross_year=244


T2_exact_build           clusters= 23 rows=  47 (1.69%)  fold_spanning=  41 (1.47%)  same_year=6 cross_year=41


T3_build_age             clusters=  8 rows=  16 (0.58%)  fold_spanning=  16 (0.58%)  same_year=0 cross_year=16


T4_full_signature        clusters=  0 rows=   0 (0.00%)  fold_spanning=   0 (0.00%)  same_year=0 cross_year=0


T4b_full_sig_no_school   clusters=  0 rows=   0 (0.00%)  fold_spanning=   0 (0.00%)  same_year=0 cross_year=0

Collapse T1 -> T4 fold-spanning rows: 220 -> 0 (7.91% -> 0.00%)
Strict (T4) duplicates: same_year_rows=0 cross_year_rows=0
Refined D2 escalation (strict fold-spanning > 2%): False

Artifacts written:
  outputs\validation\phase6a_d2_refined_tier_summary.csv True
  outputs\validation\phase6a_d2_refined_duplicate_candidates.csv True
  outputs\reports\phase6a_d2_refined_probe_report.md True
  outputs\reports\phase6a_d2_refined_experiment_log_candidate.csv True
Main log unchanged: True
Refined D2 escalation: False
